In [1]:
#Imports
import pandas as pd
import plotly.express as px
import plotly.io as pio
import google.generativeai as genai
from google.genai import types
from google import genai
import csv
import os
import numpy as np
from plotly.subplots import make_subplots 
import plotly.graph_objects as go

In [2]:
#Helper Methods
def convert_to_numbers(x):
    if isinstance(x, str):
        x = x.strip().upper()
        if x.endswith("K"):
            return float(x[:-1]) * 1000
    return pd.to_numeric(x, errors='coerce')

In [22]:
csv_files = [
    '../Data/TimingData/2021-timing.csv',
    '../Data/TimingData/2022-timing.csv',
    '../Data/TimingData/2023-timing.csv',
    '../Data/TimingData/2024-timing.csv'
]

color_map = {
    'COMPLETED': 'green',
    'FAILED': 'red',
    'NODE_FAIL': 'red',
    'TIMEOUT': 'yellow',
    'OUT_OF_MEMORY': 'orange',
    'RESIZING': 'blue',
    'REQUEUED': 'blue',
    'CANCELLED': 'black'
}

for csv_file in csv_files:
    df = pd.read_csv(csv_file)

    states = df.iloc[:, 2]
    states = states.str.replace(r'^CANCELLED.*', 'CANCELLED', regex=True)
    df['clean_state'] = states

    # Map colors
    df['color'] = df['clean_state'].map(color_map).fillna('gray')

    # Map backfilled column (column 3) to 'yes'/'no' labels for legend
    backfilled = df.iloc[:, 3].astype(str).str.lower()
    df['backfilled_label'] = backfilled.apply(lambda x: 'yes' if x == 'yes' else 'no')

    year = csv_file.split('/')[-1][:4]

    fig = px.scatter(
        df,
        x=df.columns[1],
        y=df.columns[0],
        color='clean_state',
        symbol='backfilled_label',
        color_discrete_map=color_map,
        symbol_sequence=['circle', 'cross'],
        labels={'x': 'Nodes', 'y': 'Diffsec', 'color': 'State', 'symbol': 'Backfilled'},
        title=f'Node vs Time for {year}'
    )

    fig.write_image(f'../Plots/TimingPlots/Node_vs_Diffsec{year}.png', width=1600, height=1200)

